# We will generate heatmaps for this task

The recommended heatmap is to use channel sorted with rows sorted by:
1. condition
2. timepoint
3. well location

In [15]:
import pandas as pd
from treeview_gen import df_to_treeview
from pathlib import Path
from time import sleep
import re
import os
import shutil
from natsort import natsorted

hd_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/no_treatmnet_data/no-treatment_hd_feat_red.csv")
regex_df = pd.read_csv(hd_path, index_col=0, nrows=0)
pattern = r"Region (.+?) \(global\)"

channels_found = set()
for col in regex_df.columns:
    found = re.search(pattern, col)
    if found:
        channels_found.add(found.group(1))

print(channels_found)

{'SYTO14 Rabbitts', 'TMR/AF555 Rabbitts', 'MitoTracker Deep Red', 'Alexa 488 shortened', 'HOECHST 33342 expanded'}


In [24]:
def gen_treeview_channel(df: pd.DataFrame, channel_name: str, out_dir: str, sortIndex: bool = True) -> list[str]:
    """Generates a treeviewe file set for a given channel

    Args:
        df (pd.DataFrame): channel data frame
        channel_name (str): the name of the channel
        out_dir (str): the file path where the file will be outputted to (this creates a folder)
    """
    
    out_dir_channel = os.path.join(out_dir, channel_name)
    
    if os.path.exists(out_dir_channel):
        shutil.rmtree(out_dir_channel)
        sleep(0.5)
    os.makedirs(out_dir_channel, exist_ok=True)
    
    out_dir_channel = os.path.join(out_dir_channel, f"{channel_name.replace("/","_")}_treeview")
    
    df_to_treeview(df=df, outFile=out_dir_channel, sortIndex=sortIndex, row_cluster=False)
    
    cdt_path = f"{out_dir_channel}.cdt"
    return read_tv_col_order(cdt_path)


def read_tv_col_order(file_path: str) -> list[str]:
    col_df = pd.read_csv(file_path, index_col=[0,1,2], sep="\t")
    return col_df.columns.tolist()

# lets remove the orginal channel sorted folder
def clean_folder(path: str) -> None:
    if os.path.exists(path):
        shutil.rmtree(path)
        sleep(0.5)
    os.makedirs(path, exist_ok=True)
    
def sort_keys(label):
    parts = label.split('._.')
    well = parts[0]
    timepoint = parts[1]
    
    letter = well[0]
    number = int(well[1:])
    time_val = int(re.search(r'(\d+)h', timepoint).group(1))
    
    return (letter, time_val, number)

In [34]:
channel_dir = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/no_treatmnet_data/heatmaps/channel_heatmaps")
clean_folder(channel_dir)

hd_df = pd.read_csv(hd_path, index_col=0)
hd_df = hd_df.loc[sorted(hd_df.index, key=sort_keys)]
hd_df = hd_df[~hd_df.index.str.contains(r'(?<=\D)(1|24)\._\.', regex=True)]

channel_dfs = {ch: hd_df[[col for col in hd_df.columns if ch in col]] for ch in channels_found}

col_order = []
for name, df in channel_dfs.items():
    order = gen_treeview_channel(df, name.replace("/", "_"), channel_dir)
    col_order.extend(order)
    

print(col_order)

reorder_df = hd_df[col_order]

final_out_path = Path('/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/no_treatmnet_data/heatmaps/no_treatment_SORTED-CLUSTERED_channels')
df_to_treeview(df=reorder_df, outFile=str(final_out_path), sortIndex=False, row_cluster=False, col_cluster=False)
# display(hd_df)

/tmp/ipykernel_50922/3965295866.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hd_df = hd_df[~hd_df.index.str.contains(r'(?<=\D)(1|24)\._\.', regex=True)]


['Nuclei - Ring Region SYTO14 Rabbitts (global) Threshold Compactness 60% SP-Filter', 'Nuclei - Nucleus Region SYTO14 Rabbitts (global) Symmetry 03 SER-Edge', 'Nuclei - Nucleus Region SYTO14 Rabbitts (global) Symmetry 05 SER-Edge', 'Nuclei - Membrane Region SYTO14 Rabbitts (global) Threshold Compactness 60% SP-Filter', 'Nuclei - Ring Region SYTO14 Rabbitts (global) Profile 4/5', 'Nuclei - Cell Region SYTO14 Rabbitts (global) Gabor Min 2 px w2', 'Nuclei - Cytoplasm Region SYTO14 Rabbitts (global) Gabor Min 2 px w2', 'Nuclei - Nucleus Region SYTO14 Rabbitts (global) Radial Relative Deviation SER-Valley', 'Nuclei - Nucleus Region SYTO14 Rabbitts (global) Radial Mean Ratio SP-Filter', 'Nuclei - Nucleus Region SYTO14 Rabbitts (global) Threshold Compactness 40%', 'Nuclei - Cell Region SYTO14 Rabbitts (global) Threshold Compactness 40%', 'Nuclei - Ring Region SYTO14 Rabbitts (global) Gabor Min 2 px w2', 'Nuclei - Nucleus Region SYTO14 Rabbitts (global) Radial Mean Ratio SER-Bright', 'Nuclei -